# CS 6470 A1 — Data, leakage, and a first Pipeline

**70 points** · Modules 2–3 (due Module 3) · plan for about 4–6 focused hours

**Before you submit, rename this file to `CS6470_A1_LastName_FirstName.ipynb`.**
Open it in [Google Colab](https://colab.research.google.com/) (File → Upload notebook) or local Jupyter (Python 3.10+).

## How this notebook works

- It has **6 numbered tasks**, top to bottom. Cells marked **`# YOUR CODE`** and Markdown cells marked **YOUR ANSWER** are the graded work. Cells marked *Runs as-is* are provided — run them and read them.
- Every coding task ends with a **Checkpoint** describing what you should see, so you always know whether you are on track. Small differences (±0.01–0.02 on scores) are normal.
- **Hints name every class and argument.** Newer to Python? Follow them literally. Comfortable? Try each task from the description first. Experienced? There is an ungraded stretch at the end of the assignment page.
- A `# YOUR CODE` cell raises `NotImplementedError` until you replace that line — that is the notebook telling you where to work, not a bug.
- Before submitting: **Runtime → Restart and run all.** Every cell must run with no errors and no `NotImplementedError` left.


## Setup and data *(runs as-is)*

**Dataset: Adult Census Income** (OpenML `adult`, version 2) — 48,842 U.S. census rows, 14 feature columns, target `class`: does the person earn `>50K` or `<=50K` (about 76% / 24%). Three columns have missing values (`workclass`, `occupation`, `native-country`). Two quirks matter for the leakage hunt: `fnlwgt` is a **census sampling weight**, not a fact about the person, and `education` / `education-num` encode the **same information twice**.

The cell below loads everything and imports every tool you need for this notebook. The download takes ~30 seconds the first time.


In [1]:
# Runs as-is — imports for the whole notebook + data load.
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

adult = fetch_openml("adult", version=2, as_frame=True, parser="auto")
X, y = adult.data, adult.target
print("loaded:", X.shape, "target classes:", list(y.unique()))
X.head()


loaded: (48842, 14) target classes: ['<=50K', '>50K']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


## Task 1 of 6 — Explore the data (EDA)

Produce, in code:

1. The table's **shape** and **dtypes**.
2. **Missing-value counts** per column (only some columns have any).
3. The **target balance** as shares (not raw counts).
4. **Two labeled plots**: a histogram of one numeric column split by income class (try `age` or `hours-per-week`), and a bar chart of one categorical column's value counts (try `education`).

**Hints.** `X.shape` · `X.dtypes` · `X.isna().sum()` · `y.value_counts(normalize=True)` · for the split histogram: `X[y == ">50K"]["age"].plot(kind="hist", alpha=0.5, label=">50K")` then the same for `<=50K`, then `plt.legend()`; for the bar chart: `X["education"].value_counts().plot(kind="bar")`. Call `plt.show()` after each figure and give every plot a title.

**Checkpoint.** 48,842 rows × 14 columns; exactly three columns show missing values (`workclass` 2,799 · `occupation` 2,809 · `native-country` 857); the `>50K` share is ≈ 0.24.


In [8]:
# YOUR CODE — Task 1: shape, dtypes, missing counts, target balance, two labeled plots.
X.shape
X.dtypes
X.isna().sum()
y.value_counts(normalize=True)


class
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64

### Task 1 (finish) — columns you will drop — YOUR ANSWER

**TODO:** Name the column(s) you will drop before modeling and give one sentence of reason for each. (The Setup cell described two quirks — start there.)


## Task 2 of 6 — Leakage hunt — YOUR ANSWER

**Scenario:** a bank wants this model to estimate income *at application time*, before any decision is made. Answer each question in 1–3 sentences — no code needed.

**Q1.** Which column is not a fact about the person at all, and why would a model that leans on it be learning noise?

**TODO:** your answer.

**Q2.** Which two columns encode the same information twice? Keeping both is not leakage — but which one would you keep, and why?

**TODO:** your answer.

**Q3.** Imagine the table also had a column `approved_last_month`. It would probably boost your test score — and it would still torpedo the model in production. Why? (Think about *when* that column gets its value.)

**TODO:** your answer.


## Task 3 of 6 — Split first

Test rows must stay unseen by **every** fitted step — including the scaler and the encoder, not just the model. So the split comes before anything is fit.

In code:

1. Drop the column(s) you flagged in Task 1 (at minimum `fnlwgt`).
2. Split into train and test with `test_size=0.2`, `stratify=y`, `random_state=42`.
3. Print the shapes of both splits and the class share in each (they should match).

**Hints.** `X = X.drop(columns=["fnlwgt"])` (add yours) · `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)` · `y_train.value_counts(normalize=True)`.

**Checkpoint.** 39,073 train rows / 9,769 test rows; the `>50K` share is ≈ 0.239 in both splits — that is what `stratify` buys you.


In [ ]:
# YOUR CODE — Task 3: drop flagged columns, split, verify shapes and class shares.

raise NotImplementedError("Task 3: delete this line and write your code")

## Task 4 of 6 — Preprocessing + model in one Pipeline

Build the preprocessing so it can only ever learn from training rows — by making it a step inside the Pipeline:

1. Two column lists **from `X_train`**: numeric columns and categorical columns.
2. A `ColumnTransformer` with two branches:
   - numeric: `SimpleImputer(strategy="median")` then `StandardScaler()`
   - categorical: `SimpleImputer(strategy="most_frequent")` then `OneHotEncoder(handle_unknown="ignore")`
3. A `Pipeline` named `pipe`: the transformer, then `DummyClassifier(strategy="most_frequent")`.
4. Fit `pipe` on the training split.

**Hints.** Column lists: `X_train.select_dtypes(include="number").columns.tolist()` and `exclude="number"`. Each branch is itself a small `Pipeline([("imp", ...), ("sc", ...)])`. The transformer: `ColumnTransformer([("num", <numeric branch>, num_cols), ("cat", <categorical branch>, cat_cols)])`. Then `pipe = Pipeline([("pre", pre), ("clf", DummyClassifier(strategy="most_frequent"))])`.

**Checkpoint.** `pipe.fit(X_train, y_train)` completes without error, and `pipe.predict(X_test)` returns 9,769 predictions.


In [ ]:
# YOUR CODE — Task 4: column lists, ColumnTransformer, Pipeline, fit.

raise NotImplementedError("Task 4: delete this line and write your code")

## Task 5 of 6 — Baseline first, then a real model

1. Report **macro-F1** for the fitted dummy pipeline on the **test** split. Macro-F1 averages the F1 of both classes equally, so it cannot be gamed by always predicting the majority — which is exactly what the dummy does, and why its macro-F1 is low.
2. Build the same Pipeline with `LogisticRegression(max_iter=1000)` in place of the dummy, fit it, and report its test macro-F1.
3. Print one comparison line: how much did the real model beat the floor by?

Do **not** refit anything on the test split, and do not tune against it — one honest look is all a test set is for.

**Hints.** `f1_score(y_test, pipe.predict(X_test), average="macro")` · reuse your `pre` object: `Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])`.

**Checkpoint.** Dummy macro-F1 ≈ **0.43**; logistic ≈ **0.78**. If your dummy scores near the logistic, preprocessing probably saw the test rows — revisit Task 3.


In [ ]:
# YOUR CODE — Task 5: dummy macro-F1, logistic macro-F1, one comparison line.

raise NotImplementedError("Task 5: delete this line and write your code")

## Task 6 of 6 — Memo (≤1 page) — YOUR ANSWER

Written for a **non-CS manager** — no sklearn class names, no jargon without a plain-language gloss. Replace each **TODO**.

### The leakage risk I found

**TODO:** 3–5 sentences. Name one concrete risk from this dataset (Task 2 has three candidates) and what would silently go wrong at decision time if it shipped.

### How the Pipeline prevents it

**TODO:** 2–4 sentences. Explain — to someone who has never heard the word "Pipeline" — how doing all preprocessing inside a fitted-on-training-only recipe protects the honesty of the test number.

### The number I would report

**TODO:** 2–3 sentences. Which single number would you give the manager (name it in words, not just "macro-F1"), what is its value, and what is the one caveat you would attach?

### AI disclosure

**TODO:** Tool + what it did, or "I did not use AI."


## Before you submit

- [ ] File renamed to `CS6470_A1_LastName_FirstName.ipynb`
- [ ] **Runtime → Restart and run all** — every cell ran, no errors, no `NotImplementedError` anywhere
- [ ] All 6 tasks done: every `# YOUR CODE` cell has your code, every **YOUR ANSWER** / **TODO** prompt has your words
- [ ] Memo written in the Markdown cells above (it stays inside this notebook — no separate PDF)
- [ ] AI-use disclosure filled in

Upload the `.ipynb` file to this assignment on Canvas (in Colab: File → Download → Download .ipynb).

Integrity: the notebook and memo must be yours. AI may help explain an error; it may not be the submitted analysis.
